Step 1. Reviews Bronze Layer

In [0]:
from pyspark.sql.functions import lit, current_timestamp

cities = ["austin", "chicago", "twin_cities", "nashville", "seattle"]

reviews_dfs = []

for city in cities:
    file_path = f"/Volumes/workspace/default/airbnb_raw_2/{city}/{city}_reviews.csv"
    
    try:
        df = spark.read.option("header", "true") \
                       .option("multiLine", "true") \
                       .option("escape", '"') \
                       .csv(file_path)
        
        df = df.withColumn("city", lit(city)) \
               .withColumn("source_file", lit(f"{city}_reviews.csv")) \
               .withColumn("ingestion_timestamp", current_timestamp())
        
        reviews_dfs.append(df)
        print(f"Loaded {city}")
        
    except Exception as e:
        print(f"Failed to load {city}: {e}")

if len(reviews_dfs) == 0:
    raise ValueError("No reviews files were loaded.")

df_reviews_bronze = reviews_dfs[0]

for df in reviews_dfs[1:]:
    df_reviews_bronze = df_reviews_bronze.unionByName(df, allowMissingColumns=True)

df_reviews_bronze.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/airbnb_bronze_2/reviews"
)

Loaded austin
Loaded chicago
Loaded twin_cities
Loaded nashville
Loaded seattle


Step 2. Reviews Validation

In [0]:
display(df_reviews_bronze.groupBy("city").count())

city,count
austin,588362
chicago,492465
twin_cities,300687
nashville,784894
seattle,575824


In [0]:
print(df_reviews_bronze.columns)

['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'city', 'source_file', 'ingestion_timestamp']


Step 3. Calendar Bronze Layer

In [0]:
from pyspark.sql.functions import lit, current_timestamp

cities = ["austin", "chicago", "twin_cities", "nashville", "seattle"]

calendar_dfs = []

for city in cities:
    file_path = f"/Volumes/workspace/default/airbnb_raw_2/{city}/{city}_calendar.csv"
    
    try:
        df = spark.read.option("header", "true") \
                       .option("multiLine", "true") \
                       .option("escape", '"') \
                       .csv(file_path)
        
        df = df.withColumn("city", lit(city)) \
               .withColumn("source_file", lit(f"{city}_calendar.csv")) \
               .withColumn("ingestion_timestamp", current_timestamp())
        
        calendar_dfs.append(df)
        print(f"Loaded {city}")
        
    except Exception as e:
        print(f"Failed to load {city}: {e}")

if len(calendar_dfs) == 0:
    raise ValueError("No calendar files were loaded.")

df_calendar_bronze = calendar_dfs[0]

for df in calendar_dfs[1:]:
    df_calendar_bronze = df_calendar_bronze.unionByName(df, allowMissingColumns=True)

df_calendar_bronze.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/airbnb_bronze_2/calendar"
)

Loaded austin
Loaded chicago
Loaded twin_cities
Loaded nashville
Loaded seattle


Step 4. Calendar Validation

In [0]:
display(df_calendar_bronze.groupBy("city").count())

city,count
austin,3844547
chicago,3161995
twin_cities,1941071
nashville,3446696
seattle,2553540


In [0]:
display(df_calendar_bronze.select("city", "price").groupBy("city").count())

city,count
austin,3844547
chicago,3161995
twin_cities,1941071
nashville,3446696
seattle,2553540


In [0]:
print(df_calendar_bronze.columns)

['listing_id', 'date', 'available', 'price', 'adjusted_price', 'minimum_nights', 'maximum_nights', 'city', 'source_file', 'ingestion_timestamp']


Step 5. Listings Bronze Layer

In [0]:
from pyspark.sql.functions import lit, current_timestamp

cities = ["austin", "chicago", "twin_cities", "nashville", "seattle"]

listings_dfs = []

for city in cities:
    file_path = f"/Volumes/workspace/default/airbnb_raw_2/{city}/{city}_listings.csv"
    
    try:
        df = spark.read.option("header", "true") \
                       .option("multiLine", "true") \
                       .option("escape", '"') \
                       .csv(file_path)
        
        df = df.withColumn("city", lit(city)) \
               .withColumn("source_file", lit(f"{city}_listings.csv")) \
               .withColumn("ingestion_timestamp", current_timestamp())
        
        listings_dfs.append(df)
        print(f"Loaded {city}")
        
    except Exception as e:
        print(f"Failed to load {city}: {e}")

if len(listings_dfs) == 0:
    raise ValueError("No listings files were loaded.")

df_listings_bronze = listings_dfs[0]

for df in listings_dfs[1:]:
    df_listings_bronze = df_listings_bronze.unionByName(df, allowMissingColumns=True)

df_listings_bronze.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/airbnb_bronze_2/listings"
)

Loaded austin
Loaded chicago
Loaded twin_cities
Loaded nashville
Loaded seattle


Step 7. Listings Validation

In [0]:
display(df_listings_bronze.groupBy("city").count())

city,count
austin,10533
chicago,8660
twin_cities,5318
nashville,9443
seattle,6996


In [0]:
print(df_listings_bronze.columns)

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availabil

Step 8. Final Validation

In [0]:
display(spark.read.format("delta").load("/Volumes/workspace/default/airbnb_bronze_2/listings").groupBy("city").count())
display(spark.read.format("delta").load("/Volumes/workspace/default/airbnb_bronze_2/calendar").groupBy("city").count())
display(spark.read.format("delta").load("/Volumes/workspace/default/airbnb_bronze_2/reviews").groupBy("city").count())

city,count
austin,10533
nashville,9443
chicago,8660
seattle,6996
twin_cities,5318


city,count
nashville,3446696
austin,3844547
chicago,3161995
seattle,2553540
twin_cities,1941071


city,count
nashville,784894
seattle,575824
austin,588362
chicago,492465
twin_cities,300687


In [0]:
print(spark.read.format("delta").load("/Volumes/workspace/default/airbnb_bronze_2/listings").columns)
print(spark.read.format("delta").load("/Volumes/workspace/default/airbnb_bronze_2/calendar").columns)
print(spark.read.format("delta").load("/Volumes/workspace/default/airbnb_bronze_2/reviews").columns)

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availabil